In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def cleaningHousePrices(dataSet):
    # Mantener la columna 'Id' para el envío
    id_column = dataSet['Id']

    # Imputar valores faltantes en el conjunto de datos
    # Usar la media para columnas numéricas
    for column in dataSet.select_dtypes(include=['float64', 'int64']).columns:
        dataSet[column].fillna(dataSet[column].mean(), inplace=True)

    # Mapear la columna 'Street' si existe
    if "Street" in dataSet.columns:
        dataSet["Street"] = dataSet["Street"].map({'Grvl': 0, 'Pave': 1})

    # Imputar o eliminar columnas de tipo objeto
    for i in dataSet.select_dtypes(include=["object"]).columns:
        # Convertir a tipo categórico o eliminar
        dataSet.drop(i, axis=1, inplace=True)

    # Eliminar columnas específicas si es necesario
    if "MSZoning" in dataSet.columns:
        dataSet.drop(["MSZoning"], axis=1, inplace=True)

    # Asegúrate de que el DataFrame no esté vacío
    if dataSet.empty:
        raise ValueError("El DataFrame está vacío después de la limpieza.")

    # Reincorporar la columna 'Id'
    dataSet['Id'] = id_column

    return dataSet

In [ ]:
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

In [ ]:
cleanTrain = cleaningHousePrices(train_data)
cleanTest = cleaningHousePrices(test_data)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import statsmodels.api as sm

In [ ]:
# Dividir el conjunto de entrenamiento
np.random.seed(0)
df_train, df_test = train_test_split(cleanTrain, train_size=0.7, test_size=0.3, random_state=100)

In [ ]:
# Escalado de variables numéricas
scaler = MinMaxScaler()
num_vars = ['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
            'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea',
            'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
            '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea',
            'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
            'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces',
            'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF',
            'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch',
            'PoolArea', 'MiscVal', 'MoSold', 'YrSold']

df_train[num_vars] = scaler.fit_transform(df_train[num_vars])

In [ ]:
# Dividir datos en variables X e y
y_train = df_train.pop('SalePrice')
X_train = df_train

In [ ]:
# Función para construir el modelo
def build_model(X, y):
    X = sm.add_constant(X)  # Agregar la constante
    lm = sm.OLS(y, X).fit()  # Ajustar el modelo
    print(lm.summary())  # Resumen del modelo
    return lm

In [ ]:
#print(np.isinf(X_train).sum())
print(X_train.isnull().sum())

In [ ]:
# Construir el modelo
model1 = build_model(X_train, y_train)

In [ ]:
# Preparamos el conjunto de test
cleanTest[num_vars] = scaler.transform(cleanTest[num_vars])  # Usamos el scaler entrenado


In [ ]:
# Realizamos predicciones
X_test = cleanTest  # No necesitamos 'SalePrice' en el conjunto de test
predictions = model1.predict(sm.add_constant(X_test))  # Agregamos la constante para el modelo
predictions


In [ ]:
# Creamos un nuevo DataFrame con las predicciones y el ID
results = cleanTest[['Id']].copy()  # Asegúrate de que 'Id' es la columna de identificación
results['SalePrice'] = predictions

In [ ]:
# Mostramos el DataFrame resultante
print(results)

In [ ]:
# Si quieres guardar el DataFrame como un archivo CSV
results.to_csv('predictions.csv', index=False)